# Notebook 03 — Naive Baseline

**Owner:** Satya Yadav  
**Task:** `src/metrics.py` (MAE / RMSE / R²), naive baseline, branch `feat/satya-metrics-baseline`

---

## Purpose

This notebook establishes the **performance floor** that every subsequent model must beat.
A naive baseline makes no use of the input features — it only knows the training-set
target distribution. If Random Forest or XGBoost cannot beat this floor, something has
gone wrong.

### Baselines evaluated

| Baseline | Predicts | Rationale |
| :--- | :--- | :--- |
| **Mean** | Training-set mean of `CCME_Values` | Standard regression floor; minimises MSE |
| **Median** | Training-set median of `CCME_Values` | Robust to outliers; minimises MAE |

Both baselines are evaluated on **validation** and **test** sets using the exact same
`evaluate()` call from `src/evaluate_api.py` that every other model uses.

### Rules (from README § 3)
- One shared split — loaded via `get_splits()`, never re-created here.
- All constants (mean, median) are computed from **training rows only**.
- The test set is shown here **once** for the final comparison table.
- Random seed sourced from `src/config.SEED` (= 42).

### Reference
- Target: `CCME_Values` — continuous CCME Water Quality Index (range ≈ 31.3 – 100.0).
- Metrics: MAE, RMSE, R², MAPE, Median AE — implemented in `src/metrics.py`.

---
## 0. Imports and path setup

In [ ]:
from __future__ import annotations

import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Make sure the repo root is on the path so src.* imports work
REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.config import FIGURES, REPORTS, SEED, TARGET_COLUMN
from src.evaluate_api import evaluate, get_splits
from src.metrics import calibration_data, regression_metrics

FIGURES.mkdir(parents=True, exist_ok=True)
REPORTS.mkdir(parents=True, exist_ok=True)

# Seaborn / matplotlib style — consistent with the rest of the notebooks
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.2)
plt.rcParams.update({'figure.dpi': 120, 'savefig.dpi': 150, 'figure.figsize': (9, 5)})

print(f'SEED = {SEED}  |  TARGET = {TARGET_COLUMN}')
print(f'FIGURES -> {FIGURES}')
print(f'REPORTS -> {REPORTS}')

---
## 1. Load the shared split

`get_splits()` loads `clean.parquet`, reads the saved site-grouped split indices from
`data/splits/splits.parquet`, and returns preprocessed (X_train, …, y_test) with
train-only fitted imputation.

> **Run `python -m src.split` first** if `data/splits/splits.parquet` does not exist yet.
> The parquet is git-ignored; the cleaned parquet (`data/processed/clean.parquet`) IS
> committed and should already be present after `git lfs pull`.

In [ ]:
X_train, X_val, X_test, y_train, y_val, y_test = get_splits()

print('Split sizes')
print(f'  Train  : {len(y_train):>9,} rows')
print(f'  Val    : {len(y_val):>9,} rows')
print(f'  Test   : {len(y_test):>9,} rows')
print(f'  Total  : {len(y_train) + len(y_val) + len(y_test):>9,} rows')

---
## 2. Training-set target distribution

Key statistics computed on `y_train` only — these are the only numbers the baseline
is allowed to know.

In [ ]:
TRAIN_MEAN   = float(y_train.mean())
TRAIN_MEDIAN = float(y_train.median())
TRAIN_STD    = float(y_train.std())
TRAIN_MIN    = float(y_train.min())
TRAIN_MAX    = float(y_train.max())

print('Training-set CCME_Values statistics (these define the baselines)')
print(f'  Mean   = {TRAIN_MEAN:.4f}')
print(f'  Median = {TRAIN_MEDIAN:.4f}')
print(f'  Std    = {TRAIN_STD:.4f}')
print(f'  Min    = {TRAIN_MIN:.4f}')
print(f'  Max    = {TRAIN_MAX:.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Histogram (all three splits) ---
for ax, (name, y) in zip(
    [axes[0], axes[0], axes[0]],
    [('Train', y_train), ('Val', y_val), ('Test', y_test)],
):
    ax.hist(y.values, bins=60, alpha=0.55, label=name, edgecolor='none')

axes[0].axvline(TRAIN_MEAN,   color='crimson',  lw=2, ls='--', label=f'Train mean = {TRAIN_MEAN:.1f}')
axes[0].axvline(TRAIN_MEDIAN, color='royalblue', lw=2, ls='-.',  label=f'Train median = {TRAIN_MEDIAN:.1f}')
axes[0].set_xlabel('CCME_Values')
axes[0].set_ylabel('Count')
axes[0].set_title('CCME_Values distribution — all splits')
axes[0].legend(fontsize=9)

# --- Empirical CDF ---
for name, y in [('Train', y_train), ('Val', y_val), ('Test', y_test)]:
    sorted_y = np.sort(y.values)
    axes[1].plot(sorted_y, np.linspace(0, 1, len(sorted_y)), label=name, lw=1.5)

axes[1].axvline(TRAIN_MEAN,   color='crimson',  lw=1.5, ls='--', label='Train mean')
axes[1].axvline(TRAIN_MEDIAN, color='royalblue', lw=1.5, ls='-.', label='Train median')
axes[1].set_xlabel('CCME_Values')
axes[1].set_ylabel('Empirical CDF')
axes[1].set_title('Empirical CDF — CCME_Values across splits')
axes[1].legend(fontsize=9)

plt.tight_layout()
fig.savefig(FIGURES / '03_target_distribution.png', bbox_inches='tight')
plt.show()
print(f"Saved -> {FIGURES / '03_target_distribution.png'}")

---
## 3. Naive baselines: build constant predictors

Both predictors are built strictly from training-set statistics.

In [ ]:
def constant_predictor(constant: float, n: int) -> np.ndarray:
    """Return an array of length n where every prediction equals `constant`."""
    return np.full(n, constant)


# Mean baseline: minimises MSE — directly optimised for RMSE evaluation
mean_val_pred  = constant_predictor(TRAIN_MEAN,   len(y_val))
mean_test_pred = constant_predictor(TRAIN_MEAN,   len(y_test))

# Median baseline: minimises MAE — robust alternative
med_val_pred  = constant_predictor(TRAIN_MEDIAN, len(y_val))
med_test_pred = constant_predictor(TRAIN_MEDIAN, len(y_test))

print('Constant predictors built from training statistics only.')
print(f'  Mean baseline   -> always predicts {TRAIN_MEAN:.4f}')
print(f'  Median baseline -> always predicts {TRAIN_MEDIAN:.4f}')

---
## 4. Evaluate baselines using the shared `evaluate()` API

The same `evaluate(y_true, y_pred)` call used by Random Forest and XGBoost.

In [ ]:
results = {}

for name, val_pred, test_pred in [
    ('Mean baseline',   mean_val_pred,  mean_test_pred),
    ('Median baseline', med_val_pred,   med_test_pred),
]:
    val_metrics  = evaluate(y_val,  val_pred)
    test_metrics = evaluate(y_test, test_pred)
    results[name] = {'val': val_metrics, 'test': test_metrics}

    print(f'\n=== {name} ===')
    print(f"  Val  → MAE={val_metrics['mae']:.4f}  RMSE={val_metrics['rmse']:.4f}  R²={val_metrics['r2']:.6f}  MAPE={val_metrics['mape']:.2f}%  MedAE={val_metrics['medae']:.4f}")
    print(f"  Test → MAE={test_metrics['mae']:.4f}  RMSE={test_metrics['rmse']:.4f}  R²={test_metrics['r2']:.6f}  MAPE={test_metrics['mape']:.2f}%  MedAE={test_metrics['medae']:.4f}")

**Expected behaviour:**
- A constant predictor always yields **R² ≈ 0** (or slightly negative if val/test have
  different variance from train) because it has zero explanatory power.
- The mean baseline minimises RMSE among all constant predictors.
- The median baseline minimises MAE among all constant predictors.

Both set a hard floor: any model with R² ≤ 0 or MAE ≥ these values has failed.

---
## 5. Results table

In [ ]:
rows = []
for model_name, splits_dict in results.items():
    for split_name, metrics in splits_dict.items():
        row = {'Model': model_name, 'Split': split_name}
        row.update(metrics)
        rows.append(row)

results_df = pd.DataFrame(rows)
display_cols = ['Model', 'Split', 'mae', 'rmse', 'r2', 'mape', 'medae']
results_df = results_df[display_cols]

# Round for display
for col in ['mae', 'rmse', 'r2', 'mape', 'medae']:
    results_df[col] = results_df[col].round(4)

print('\n=== Naive Baseline Results ===')
print(results_df.to_string(index=False))

In [ ]:
# Save to reports/ so the final evaluation notebook can load it
out_path = REPORTS / 'naive_baseline_results.csv'
results_df.to_csv(out_path, index=False)
print(f'Saved -> {out_path}')

---
## 6. Residual analysis: where does the constant prediction fail?

Plotting residuals (y_true − y_pred) for the mean baseline on the validation set reveals
which parts of the target range the baseline systematically over- or under-estimates.
This motivates why a model that uses the features should do better.

In [ ]:
residuals_val = y_val.values - mean_val_pred

fig, axes = plt.subplots(1, 3, figsize=(17, 5))

# --- Residual histogram ---
axes[0].hist(residuals_val, bins=60, color='steelblue', edgecolor='none', alpha=0.85)
axes[0].axvline(0, color='crimson', lw=2, ls='--', label='Zero residual')
axes[0].set_xlabel('Residual  (y_true − y_pred)')
axes[0].set_ylabel('Count')
axes[0].set_title('Residual distribution — mean baseline (val)')
axes[0].legend()

# --- Predicted vs Actual ---
sample_idx = np.random.default_rng(SEED).choice(len(y_val), size=min(5000, len(y_val)), replace=False)
axes[1].scatter(
    mean_val_pred[sample_idx], y_val.values[sample_idx],
    alpha=0.15, s=6, color='steelblue',
)
lims = [y_val.min(), y_val.max()]
axes[1].plot(lims, lims, 'r--', lw=1.5, label='Perfect prediction')
axes[1].set_xlabel('Predicted (constant mean)')
axes[1].set_ylabel('Actual CCME_Values')
axes[1].set_title('Predicted vs Actual — mean baseline (val)')
axes[1].legend()

# --- Residual vs Actual (shows structure the baseline misses) ---
axes[2].scatter(
    y_val.values[sample_idx], residuals_val[sample_idx],
    alpha=0.15, s=6, color='darkorange',
)
axes[2].axhline(0, color='crimson', lw=1.5, ls='--')
axes[2].set_xlabel('Actual CCME_Values')
axes[2].set_ylabel('Residual')
axes[2].set_title('Residual vs Actual — structure the baseline misses')

plt.tight_layout()
fig.savefig(FIGURES / '03_baseline_residuals.png', bbox_inches='tight')
plt.show()
print(f"Saved -> {FIGURES / '03_baseline_residuals.png'}")

**Interpretation of the residual plot (right panel):**

- For low true values (< mean), the baseline over-predicts → negative residuals.
- For high true values (> mean), the baseline under-predicts → positive residuals.
- The residual-vs-actual plot shows a strong linear trend — precisely the pattern
  a trained regression model can capture using the chemical measurements.

---
## 7. CCME category calibration

As a secondary diagnostic, bin both ground-truth and baseline predictions into the
five CCME WQI categories and compute macro F1. This shows how badly a constant
predictor fails on the classification view of the same problem.

In [ ]:
print('=== CCME category calibration — mean baseline on val set ===')
cal_result = calibration_data(y_val.values, mean_val_pred)
cls_m = cal_result['classification_metrics']

print(f"  Accuracy  : {cls_m['accuracy']:.4f}")
print(f"  Macro F1  : {cls_m['macro_f1']:.4f}")
print(f"  Weighted F1: {cls_m['weighted_f1']:.4f}")
print()
print('  Per-class F1:')
for cls, f1 in cls_m['per_class_f1'].items():
    sup = cls_m['support'][cls]
    print(f'    {cls:<12} F1={f1:.4f}   support={sup:,}')

In [ ]:
# Confusion matrix — manual computation for display
CCME_THRESHOLDS = {
    'Poor': (0.0, 44.0),
    'Marginal': (45.0, 64.0),
    'Fair': (65.0, 79.0),
    'Good': (80.0, 94.0),
    'Excellent': (95.0, 100.0),
}

def bin_ccme(values: np.ndarray) -> np.ndarray:
    out = np.full(len(values), 'Unknown', dtype=object)
    for label, (lo, hi) in CCME_THRESHOLDS.items():
        out[(values >= lo) & (values <= hi)] = label
    return out

ordered_cats = list(CCME_THRESHOLDS.keys())
y_val_cats   = bin_ccme(y_val.values)
pred_cats_mean = bin_ccme(mean_val_pred)

# Count distribution
val_counts  = {c: int(np.sum(y_val_cats == c))       for c in ordered_cats}
pred_counts = {c: int(np.sum(pred_cats_mean == c))   for c in ordered_cats}

print('\nVal set true category distribution:')
for cat, cnt in val_counts.items():
    pct = 100 * cnt / len(y_val)
    print(f'  {cat:<12} {cnt:>8,}  ({pct:5.1f} %)')

print('\nMean baseline predicted category distribution:')
for cat, cnt in pred_counts.items():
    pct = 100 * cnt / len(y_val)
    print(f'  {cat:<12} {cnt:>8,}  ({pct:5.1f} %)')

In [ ]:
# Bar chart: true vs predicted category counts
x = np.arange(len(ordered_cats))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x - width/2, [val_counts[c]  for c in ordered_cats], width, label='True',
       color='steelblue', alpha=0.85)
ax.bar(x + width/2, [pred_counts[c] for c in ordered_cats], width, label='Mean baseline predicted',
       color='darkorange', alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels(ordered_cats)
ax.set_xlabel('CCME WQI Category')
ax.set_ylabel('Count (val set)')
ax.set_title('True vs Predicted CCME category counts — mean baseline')
ax.legend()
plt.tight_layout()
fig.savefig(FIGURES / '03_category_counts.png', bbox_inches='tight')
plt.show()
print(f"Saved -> {FIGURES / '03_category_counts.png'}")

---
## 8. Summary and floor values

These are the **performance floors** every model must beat on the **test set**.

In [ ]:
mean_test_m = results['Mean baseline']['test']
med_test_m  = results['Median baseline']['test']

summary = pd.DataFrame({
    'Model': ['Mean baseline', 'Median baseline'],
    'MAE':   [round(mean_test_m['mae'],  4), round(med_test_m['mae'],  4)],
    'RMSE':  [round(mean_test_m['rmse'], 4), round(med_test_m['rmse'], 4)],
    'R²':    [round(mean_test_m['r2'],   6), round(med_test_m['r2'],   6)],
    'MAPE':  [round(mean_test_m['mape'], 2), round(med_test_m['mape'], 2)],
    'MedAE': [round(mean_test_m['medae'],4), round(med_test_m['medae'],4)],
})

print('=== PERFORMANCE FLOORS (test set) ===')
print(summary.to_string(index=False))
print()
print('Every model trained on features must achieve on the test set:')
print(f'  MAE  < {mean_test_m["mae"]:.4f}   (beat mean baseline)')
print(f'  RMSE < {mean_test_m["rmse"]:.4f}   (beat mean baseline)')
print(f'  R²   > 0.0000   (must explain more than zero variance)')

---
## 9. Notebook checklist

- [x] `TRAIN_MEAN` and `TRAIN_MEDIAN` computed from **training rows only**.
- [x] Both baselines evaluated via the shared `evaluate()` from `src/evaluate_api.py`.
- [x] Test set shown once, for the final results table.
- [x] Results saved to `reports/naive_baseline_results.csv`.
- [x] Residual plot saved to `reports/figures/03_baseline_residuals.png`.
- [x] Category calibration computed via `src.metrics.calibration_data()`.
- [x] Zero model parameters, zero feature information used — pure constant predictors.